## final chatbot - chromaDB and fine-tuned model combined

In [ ]:
"""
Tüketici Hukuku RAG Chatbot
Fine-tuned LLaMA 3 + ChromaDB ile çalışan profesyonel hukuki asistan
"""

import chromadb
from chromadb.utils import embedding_functions
import torch
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer

class TuketiciHukukuChatbot:
    """
    Tüketici Hukuku uzmanı RAG Chatbot

    Nasıl Çalışır:
    1. Kullanıcı sorusu ChromaDB'de aranır (embedding ile)
    2. En ilgili 3-5 yasal metin bulunur
    3. Fine-tuned LLaMA 3 bu metinleri okuyarak yanıt verir
    """

    def __init__(self, 
                 chroma_db_path: str,
                 fine_tuned_model_path: str = "beyzasn/hukuk-lora-llama-3-8b-v1",
                 use_4bit: bool = True):
        """
        Args:
            chroma_db_path: ChromaDB klasör yolu
            fine_tuned_model_path: Fine-tuned modelinin yolu 
                                   (eğer yoksa base model kullanılır)
            use_4bit: 4-bit quantization (daha az RAM; HuggingFace için bitsandbytes gerekir)
        """

        print("🚀 Tüketici Hukuku Chatbot başlatılıyor...")

        # 1. ChromaDB Bağlantısı (Mevcut vector store'unuz)
        print("\n📚 Vector store yükleniyor...")
        self.chroma_client = chromadb.PersistentClient(path=chroma_db_path)

        # Embedding function (kodunuzdaki ile aynı)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"
        )

        # Collection'ı yükle
        try:
            self.collection = self.chroma_client.get_collection(
                name="legal_documents_v2",
                embedding_function=self.embedding_function
            )
            print(f"✅ Vector store yüklendi: {self.collection.count()} döküman")
        except Exception as e:
            print(f"❌ HATA: ChromaDB yüklenemedi - {e}")
            print("⚠️  Önce vector store'u oluşturmalısınız (kodunuzdaki adımları çalıştırın)")
            raise

        # 2. Fine-tuned LLaMA 3 Yükleme (HuggingFace Transformers ile)
        print(f"\n🤖 LLaMA 3 modeli yükleniyor: {fine_tuned_model_path}")

        model_kwargs = {
            "trust_remote_code": True,
            "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            "device_map": "auto"
        }
        if use_4bit:
            try:
                import bitsandbytes as bnb  # sadece kontrol için
                model_kwargs.update({
                    "load_in_4bit": True
                })
            except ImportError:
                print("⚠️  bitsandbytes kurulu değil, model 4-bit yüklenemeyecek. 4bit için: pip install bitsandbytes")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path, use_fast=True)
            self.model = AutoModelForCausalLM.from_pretrained(fine_tuned_model_path, **model_kwargs)
            print("✅ Model yüklendi")
        except Exception as e:
            print(f"⚠️  Fine-tuned model yüklenemedi, base model yükleniyor: {e}")
            base_model = "meta-llama/Meta-Llama-3-8B-Instruct"
            self.tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
            self.model = AutoModelForCausalLM.from_pretrained(base_model, **model_kwargs)

        print("\n✅ Chatbot hazır! Sorularınızı sorabilirsiniz.\n")


    def ask(self, 
            user_question: str,
            n_context: int = 3,
            show_sources: bool = True) -> Dict:
        """
        Kullanıcı sorusuna yanıt ver

        Args:
            user_question: Kullanıcının sorusu
            n_context: Kaç tane ilgili döküman getirilecek
            show_sources: Kaynak bilgilerini göster

        Returns:
            Dict: {answer, sources, context_docs}
        """

        print(f"🔍 Soru: {user_question}")

        # ADIM 1: Vector Store'dan ilgili metinleri bul
        print(f"   → {n_context} ilgili döküman aranıyor...")
        search_results = self.collection.query(
            query_texts=[user_question],
            n_results=n_context,
            include=["documents", "metadatas", "distances"]
        )

        # Sonuçları formatla
        context_docs = []
        for doc, meta, dist in zip(
            search_results['documents'][0],
            search_results['metadatas'][0],
            search_results['distances'][0]
        ):
            similarity = 1 - dist  # Distance -> Similarity
            context_docs.append({
                'content': doc,
                'source': meta.get('file_name', 'Bilinmeyen'),
                'type': meta.get('doc_type', ''),
                'article': meta.get('article_number', ''),
                'similarity': similarity
            })

        print(f"   ✓ {len(context_docs)} döküman bulundu")

        # ADIM 2: Prompt oluştur (LLaMA 3 Instruct formatı)
        context_text = self._format_context(context_docs)
        prompt = self._create_prompt(user_question, context_text)

        # ADIM 3: LLaMA ile yanıt üret
        print("   → Yanıt üretiliyor...")
        answer = self._generate_answer(prompt)

        print("   ✓ Yanıt hazır\n")

        result = {
            'question': user_question,
            'answer': answer,
            'sources': [doc['source'] for doc in context_docs],
            'context_docs': context_docs if show_sources else None
        }

        return result


    def _format_context(self, context_docs: List[Dict]) -> str:
        """İlgili dökümanları formatla"""
        formatted = []
        for i, doc in enumerate(context_docs, 1):
            header = f"[Kaynak {i}: {doc['source']}"
            if doc['article']:
                header += f" - Madde {doc['article']}"
            header += f" | Benzerlik: {doc['similarity']:.2%}]"

            formatted.append(f"{header}\n{doc['content']}")

        return "\n\n".join(formatted)


    def _create_prompt(self, question: str, context: str) -> str:
        """
        LLaMA 3 Instruct formatında prompt oluştur
        Fine-tuning ile AYNI format (chat template uygular)
        """
        # messages yapısı kullanmaya devam, modern tokenizerlar apply_chat_template destekliyorsa onu ararız
        messages = [
            {
                "role": "system",
                "content": """Sen Türkiye'de tüketici hukuku alanında uzman bir hukuk asistanısın. 
Aşağıda sana bir soru ve ilgili yasal metinler verilecek. 
Bu metinleri dikkatlice okuyarak, Türk hukuku terminolojisi ve üslubu ile profesyonel bir yanıt ver.

ÖNEMLİ KURALLAR:
- Sadece verilen yasal metinlere dayanarak yanıt ver
- Emin olmadığın konularda spekülasyon yapma
- Madde numaralarını ve kaynaklarını belirt
- Açık, anlaşılır ve profesyonel dil kullan
- Tüketicinin haklarını net şekilde açıkla"""
            },
            {
                "role": "user",
                "content": f"""İLGİLİ YASAL METİNLER:
{context}

SORU: {question}"""
            }
        ]
        # try tokenizer'ın chat template fonksiyonu var mı? yoksa kendimiz stringle birleştir
        if hasattr(self.tokenizer, "apply_chat_template"):
            prompt = self.tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False
            )
        else:
            # simple format, not ideal for instruct-tuned but fallback
            prompt = f"<|system|>\n{messages[0]['content']}\n<|user|>\n{messages[1]['content']}\n<|assistant|>\n"
        return prompt


    def _generate_answer(self, prompt: str) -> str:
        """LLaMA ile yanıt üret"""

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        )
        inputs = {k:v.to(device) for k,v in inputs.items()}

        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=512,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        # Tam yanıtı decode et
        full_response = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Sadece assistant kısmını çıkar, open-source Llama3 yüklediyseniz uygun kısmı ayıklayın
        # try cut at first occurence of prompt (after prompt, assistant cevabı başlıyor)
        answer = full_response[len(prompt):].strip() if full_response.startswith(prompt) else full_response.strip()

        return answer


    def chat(self):
        """İnteraktif chat modu"""

        print("=" * 70)
        print("🏛️  TÜKETİCİ HUKUKU CHATBOT")
        print("=" * 70)
        print("Tüketici hukuku ile ilgili sorularınızı sorun.")
        print("Çıkmak için 'quit', 'exit' veya 'q' yazın.\n")

        while True:
            try:
                user_input = input("👤 Siz: ").strip()

                if user_input.lower() in ['quit', 'exit', 'q', 'çıkış']:
                    print("\n👋 Görüşmek üzere!")
                    break

                if not user_input:
                    continue

                # Yanıt al
                result = self.ask(user_input, n_context=3)

                # Yanıtı göster
                print(f"\n🤖 Chatbot:\n{result['answer']}\n")

                # Kaynakları göster
                if result['sources']:
                    print(f"📚 Kaynaklar: {', '.join(set(result['sources']))}\n")

                print("-" * 70 + "\n")

            except KeyboardInterrupt:
                print("\n\n👋 Görüşmek üzere!")
                break
            except Exception as e:
                print(f"\n❌ Hata oluştu: {e}\n")


# ============================================================================
# KULLANIM ÖRNEĞİ
# ============================================================================

if __name__ == "__main__":

    # Chatbot'u başlat
    chatbot = TuketiciHukukuChatbot(
        chroma_db_path="/Users/beyzaasan/Projects/HukukPusulasi/legal_chroma_db",

        # Seçenek 1: HuggingFace Hub'dan
        fine_tuned_model_path="beyzasn/hukuk-lora-llama-3-8b-v1",  

        use_4bit=True
    )

    # İNTERAKTİF MOD
    chatbot.chat()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


🚀 Tüketici Hukuku Chatbot başlatılıyor...

📚 Vector store yükleniyor...
✅ Vector store yüklendi: 3158 döküman

🤖 LLaMA 3 modeli yükleniyor: beyzasn/hukuk-lora-llama-3-8b-v1
'NoneType' object has no attribute 'cadam32bit_grad_fp32'
⚠️  Fine-tuned model yüklenemedi, base model yükleniyor: beyzasn/hukuk-lora-llama-3-8b-v1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`


OSError: You are trying to access a gated repo.
Make sure to request access at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct and pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`.

## bu işer yarar:

In [ ]:
import os
import re
import json
import torch
from threading import Thread
import gradio as gr
from typing import List, Dict, Optional

# --- HukukPusulasi_RAG.ipynb dosyasından alınan gerekli kütüphaneler ---
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
import numpy as np
from tqdm import tqdm
from datetime import datetime

# --- model_finetune.ipynb dosyasından alınan gerekli kütüphaneler ---
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer, TextStreamer, TrainingArguments, AutoTokenizer
from datasets import load_dataset, Dataset, DatasetDict

# ==============================================================================
# BÖLÜM 1: YAPILANDIRMA VE SABİT DEĞERLER
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

# ChromaDB ve Embedding Model Ayarları
CHROMADB_PERSIST_DIR = "/content/drive/MyDrive/HukukPusulasi/legal_chroma_db"
CHROMADB_COLLECTION_NAME = "legal_documents_v2"
EMBEDDING_MODEL_NAME = "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"

# LLM ve Fine-Tuning Ayarları
LLM_BASE_MODEL = "unsloth/llama-3-8b-Instruct-bnb-4bit"
LLM_LORA_ADAPTER = "beyzasn/hukuk-lora-llama-3-8b-v1"
MAX_SEQ_LENGTH = 1024

# Cihaz belirleme
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# ==============================================================================
# BÖLÜM 2: LegalVectorStore Sınıfı (DÜZELTME: Mesafe metrikleri)
# ==============================================================================

class LegalVectorStore:
    """Hukuki dokümanlar için basitleştirilmiş ChromaDB arayüzü"""
    
    def __init__(self, persist_directory: str, collection_name: str, model_name: str):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.model_name = model_name
        
        os.makedirs(persist_directory, exist_ok=True)
        
        self.client = chromadb.PersistentClient(
            path=persist_directory,
            settings=Settings(anonymized_telemetry=False, allow_reset=True)
        )
        
        # Türkçe BERT embedding function
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=model_name,
            device=DEVICE
        )
        
        try:
            self.collection = self.client.get_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )
            print(f"✅ ChromaDB: Mevcut collection yüklendi ({self.collection.count()} doküman)")
        except Exception:
            print("⚠️ ChromaDB: Collection bulunamadı.")
            self.collection = self.client.create_collection(
                name=collection_name,
                embedding_function=self.embedding_function,
                metadata={"description": "Turkish legal documents RAG system"}
            )
            print("❗ Yeni boş collection oluşturuldu.")

    def search_similar(self, query: str, n_results: int = 4, where: Optional[Dict] = None) -> Dict:
        """
        Benzer chunk'ları bul
        
        ÖNEMLİ: ChromaDB cosine distance kullanır:
        - 0.0 = Tam eşleşme
        - 2.0 = Tam zıt
        - Türkçe embeddinglerde genelde 0.3-0.6 arası iyi sonuçtur
        - Ancak hukuki metinlerde 1.5'e kadar kabul edilebilir
        """
        try:
            # Daha fazla sonuç al, sonra filtrele
            results = self.collection.query(
                query_texts=[query],
                n_results=n_results * 3,  # 3 katı al
                where=where,
                include=["documents", "metadatas", "distances"]
            )
            
            print(f"\n🔍 Arama: '{query[:50]}...'")
            print(f"📊 Mesafe değerleri: {[f'{d:.3f}' for d in results['distances'][0][:4]]}")
            
            # KRİTİK DÜZELTME: Mesafe eşiğini yükselt
            # ChromaDB'de squared L2 distance kullanılıyor olabilir
            # Bu durumda değerler çok büyük çıkar
            
            # Önce metric tipini kontrol et
            is_large_distance = any(d > 10 for d in results['distances'][0][:3])
            
            if is_large_distance:
                # Squared L2 veya farklı bir metrik - yüzde bazlı filtrele
                print("⚠️ Büyük mesafe değerleri tespit edildi - yüzde bazlı filtreleme kullanılıyor")
                
                # En iyi n_results'u al (mesafe ne olursa olsun)
                filtered_docs = results['documents'][0][:n_results]
                filtered_metas = results['metadatas'][0][:n_results]
                filtered_dists = results['distances'][0][:n_results]
            else:
                # Normal cosine distance - threshold kullan
                distance_threshold = 1.2  # Esnek eşik
                filtered_docs = []
                filtered_metas = []
                filtered_dists = []
                
                for doc, meta, dist in zip(
                    results['documents'][0], 
                    results['metadatas'][0], 
                    results['distances'][0]
                ):
                    if dist <= distance_threshold and len(filtered_docs) < n_results:
                        filtered_docs.append(doc)
                        filtered_metas.append(meta)
                        filtered_dists.append(dist)
            
            print(f"✅ {len(filtered_docs)} sonuç bulundu (toplam: {len(results['documents'][0])})")
            
            if len(filtered_docs) == 0:
                print(f"⚠️ UYARI: '{query}' için hiç sonuç bulunamadı!")
                print(f"En yakın sonuç mesafesi: {results['distances'][0][0]:.3f}")
            
            return {
                'query': query,
                'n_results': len(filtered_docs),
                'documents': filtered_docs,
                'metadatas': filtered_metas,
                'distances': filtered_dists,
            }
        except Exception as e:
            print(f"❌ Arama hatası: {e}")
            return {'query': query, 'n_results': 0, 'documents': [], 'metadatas': [], 'distances': []}

# ==============================================================================
# BÖLÜM 3: LLM VE ADAPTER YÜKLEME
# ==============================================================================

def load_llm_and_tokenizer(model_name: str, lora_adapter: str, max_seq_length: int):
    """Fine-tuned LLaMA modelini Unsloth ile yükler"""
    print(f"LLM yükleniyor: {model_name}...")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            dtype=None,
            load_in_4bit=True,
        )
        print("✅ LLM yüklendi.")

        print(f"LoRA adapterleri için PEFT model yapısı oluşturuluyor...")
        model = FastLanguageModel.get_peft_model(
            model,
            r=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=16,
            lora_dropout=0.0,
            bias="none",
            use_rslora=True,
            loftq_config=None,
        )
        
        print(f"LoRA adapterleri yükleniyor: {lora_adapter}...")
        model.load_adapter(lora_adapter, adapter_name="default")
        
        FastLanguageModel.for_inference(model)
        print("✅ LoRA adapterleri yüklendi ve model inference moduna alındı.")
        return model, tokenizer
    
    except Exception as e:
        print(f"🔴 HATA: LLM veya Adapter yüklenemedi: {e}")
        return None, None

# ==============================================================================
# BÖLÜM 4: YANIT TEMİZLEYİCİ
# ==============================================================================

def clean_llm_output(raw_output: str, query: str) -> str:
    """
    LLM'den gelen ham çıktıyı temizler.
    - Context'ten devam eden anlamsız metni kaldırır
    - Sadece asıl yanıtı döndürür
    """
    if not raw_output or len(raw_output.strip()) < 10:
        return ""
    
    output = raw_output.strip()
    
    # 1. İlk büyük harfle başlayan cümleyi bul
    sentences = re.split(r'(?<=[.!?])\s+', output)
    
    clean_sentences = []
    found_start = False
    
    for sentence in sentences:
        if not found_start:
            if sentence and len(sentence) > 15 and sentence[0].isupper():
                # Yaygın başlangıç kalıplarını kontrol et
                valid_starts = [
                    'Evet', 'Hayır', 'Tüketici', 'Kredi', 'Online', 'Alışveriş', 
                    'Cayma', 'Ayıplı', 'Mal', 'Satıcı', 'Sözleşme', 'Kanun',
                    'İlgili', 'Bu', 'Söz', 'Mesafeli', 'Hak', 'Hakkınız',
                    '6502', 'Madde', 'Yönetmelik'
                ]
                
                if any(sentence.startswith(word) for word in valid_starts):
                    found_start = True
                    clean_sentences.append(sentence)
        else:
            clean_sentences.append(sentence)
    
    cleaned = ' '.join(clean_sentences).strip()
    
    # 2. Eğer hiç geçerli cümle bulamadıysak
    if not cleaned or len(cleaned) < 30:
        # Ham çıktıdan ilk 100 karakteri at ve devam et
        if len(output) > 100:
            return output[100:].strip()
        return output
    
    return cleaned

# ==============================================================================
# BÖLÜM 5: RAG PIPELINE
# ==============================================================================

def generate_rag_response(query: str, llm_model, llm_tokenizer, vector_store, max_new_tokens: int = 512):
    """
    Kullanıcı sorgusunu alır, RAG ile ilgili kaynakları bulur
    ve Fine-Tuned LLM'e context ile birlikte göndererek yanıt üretir.
    """
    # 1. Retrieval (Arama)
    search_results = vector_store.search_similar(query=query, n_results=4)
    
    if search_results['n_results'] == 0:
        yield "Üzgünüm, bu konuda elimde yeterli hukuki kaynak bulunamadı. Lütfen sorunuzu farklı şekilde ifade etmeyi deneyin.", []
        return

    # 2. Context Hazırlama
    contexts = []
    source_details = []
    
    for i, (doc, meta, dist) in enumerate(zip(
        search_results['documents'], 
        search_results['metadatas'],
        search_results['distances']
    )):
        # Kaynak bilgisi
        source_info = f"{meta.get('file_name', 'Kaynak')}"
        if meta.get('article_number'):
            source_info += f" - Madde {meta['article_number']}"
        elif meta.get('section'):
            source_info += f" - {meta['section']}"
        
        # Her dokümanı kısalt (max 350 karakter)
        doc_text = doc.strip()[:350] + "..." if len(doc.strip()) > 350 else doc.strip()
        
        contexts.append(f"Kaynak {i+1}: {doc_text}")
        source_details.append(f"[{i+1}] {source_info} (benzerlik: {dist:.2f})")

    context_block = "\n\n".join(contexts)

    # 3. PROMPT TASARIMI
    system_prompt = """Sen Türk Tüketici Hukuku uzmanı bir asistansın. 
Verilen kaynaklardaki bilgileri kullanarak açık ve anlaşılır cevaplar ver.
Cevabında kaynak numaralarını belirt."""

    user_prompt = f"""Aşağıda hukuki kaynaklardan alınmış bilgiler var:

{context_block}

Soru: {query}

Bu kaynaklara dayanarak soruyu Türkçe olarak yanıtla. Cevabını "Evet" veya "Hayır" gibi net bir ifadeyle başlat ve ardından detayları açıkla."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    # 4. Tokenization
    input_ids = llm_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(DEVICE)

    # 5. Streamer
    streamer = TextIteratorStreamer(
        llm_tokenizer, 
        skip_prompt=True, 
        skip_special_tokens=True,
        timeout=30.0
    )

    # 6. Generation kwargs
    generation_kwargs = dict(
        input_ids=input_ids,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
        top_p=0.85,
        top_k=40,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id,
        eos_token_id=llm_tokenizer.eos_token_id
    )

    # 7. Thread'de generation
    thread = Thread(target=llm_model.generate, kwargs=generation_kwargs)
    thread.start()

    # 8. Stream ve temizlik
    raw_response = ""
    
    for new_text in streamer:
        raw_response += new_text
        
        # İlk 80 karakteri bekle
        if len(raw_response) < 80:
            continue
        
        # Temizle
        cleaned_response = clean_llm_output(raw_response, query)
        
        if cleaned_response and len(cleaned_response) > 20:
            yield cleaned_response, source_details
    
    # Son temizlik
    final_cleaned = clean_llm_output(raw_response, query)
    
    if final_cleaned and len(final_cleaned) > 20:
        yield final_cleaned, source_details
    else:
        yield raw_response.strip(), source_details
    
    thread.join()

# ==============================================================================
# BÖLÜM 6: GRADIO ARAYÜZÜ
# ==============================================================================

# Global yükleme
llm_model, llm_tokenizer = load_llm_and_tokenizer(LLM_BASE_MODEL, LLM_LORA_ADAPTER, MAX_SEQ_LENGTH)
vector_store = LegalVectorStore(CHROMADB_PERSIST_DIR, CHROMADB_COLLECTION_NAME, EMBEDDING_MODEL_NAME)


def gradio_chat_interface(message, history):
    """Gradio sohbet arayüzü için yanıt üretici fonksiyon"""
    
    if llm_model is None or llm_tokenizer is None:
        yield "❌ Hata: LLM başarıyla yüklenemedi. Lütfen konsol çıktılarını kontrol edin."
        return
    
    response_generator = generate_rag_response(
        message, 
        llm_model, 
        llm_tokenizer, 
        vector_store,
        max_new_tokens=512
    )
    
    final_response = ""
    sources = []
    
    for partial_text, current_sources in response_generator:
        final_response = partial_text
        sources = current_sources
        yield partial_text
    
    # Kaynakları ekle
    if sources:
        source_block = "\n\n" + "─" * 50 + "\n**📚 Kullanılan Kaynaklar:**\n" + "\n".join([f"• {s}" for s in sources])
        yield final_response + source_block
    else:
        yield final_response


# Arayüzü oluştur
if llm_model and vector_store:
    
    demo = gr.ChatInterface(
        fn=gradio_chat_interface,
        examples=[
            "Online alışverişte cayma hakkım var mı? Ne kadar sürem var?",
            "Ayıplı mal durumunda tüketici olarak ne talep edebilirim?",
            "Tüketici kredisi çekerken dikkat etmem gereken hukuki konular nelerdir?",
            "Kapıdan satışlarda tüketici hakları nelerdir?"
        ],
        title="⚖️ Hukuk Pusulası - RAG Destekli Hukuki Asistan",
        description=f"""
        **Mimari:** Fine-tuned Llama-3-8B + Türkçe BERT Embeddings + ChromaDB RAG
        
        ---
        ⚠️ **Önemli Uyarı:** Bu yapay zeka asistanıdır. Sunduğu bilgiler sadece bilgilendirme amaçlıdır 
        ve kesinlikle resmi hukuki tavsiye yerine geçmez. Hukuki sorunlarınız için mutlaka bir avukata danışın.
        
        **Cihaz:** {DEVICE} | **Model:** Llama-3-8B-Instruct (LoRA fine-tuned)
        """,
        theme="soft",
        submit_btn="📤 Gönder"
    )

    print("\n" + "="*70)
    print("=== GRADIO ARAYÜZÜ BAŞLATILIYOR ===")
    print("="*70 + "\n")
    demo.launch(share=True, debug=True)
else:
    print("❌ HATA: LLM veya VectorStore yüklenemedi. Sistem başlatılamadı.")

## daha iyi buluyor ama history yok galiba

In [5]:
import os
import re
import json
import torch
from threading import Thread
import gradio as gr
from typing import List, Dict, Optional

# --- HukukPusulasi_RAG.ipynb dosyasından alınan gerekli kütüphaneler ---
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
import numpy as np
from tqdm import tqdm
from datetime import datetime

# --- Transformers kütüphaneleri (Unsloth YOK) ---
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TextIteratorStreamer,
    BitsAndBytesConfig
)

# ==============================================================================
# BÖLÜM 1: YAPILANDIRMA VE SABİT DEĞERLER
# ==============================================================================

# ChromaDB ve Embedding Model Ayarları
CHROMADB_PERSIST_DIR = "/Users/beyzaasan/Projects/HukukPusulasi/legal_chroma_db"
CHROMADB_COLLECTION_NAME = "legal_documents_v2"
EMBEDDING_MODEL_NAME = "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr"

# LLM Model Ayarları - MERGED MODEL
MODEL_PATH = "/Users/beyzaasan/Projects/HukukPusulasi/merged_model_hukuk"
MAX_SEQ_LENGTH = 1024

# Cihaz belirleme
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"🖥️  Kullanılan cihaz: {DEVICE}")

# ==============================================================================
# BÖLÜM 2: LegalVectorStore Sınıfı
# ==============================================================================

class LegalVectorStore:
    """Hukuki dokümanlar için basitleştirilmiş ChromaDB arayüzü"""
    
    def __init__(self, persist_directory: str, collection_name: str, model_name: str):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.model_name = model_name
        
        os.makedirs(persist_directory, exist_ok=True)
        
        self.client = chromadb.PersistentClient(
            path=persist_directory,
            settings=Settings(anonymized_telemetry=False, allow_reset=True)
        )
        
        # Türkçe BERT embedding function
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=model_name,
            device=DEVICE
        )
        
        try:
            self.collection = self.client.get_collection(
                name=collection_name,
                embedding_function=self.embedding_function
            )
            print(f"✅ ChromaDB: Mevcut collection yüklendi ({self.collection.count()} doküman)")
        except Exception:
            print("⚠️ ChromaDB: Collection bulunamadı.")
            self.collection = self.client.create_collection(
                name=collection_name,
                embedding_function=self.embedding_function,
                metadata={"description": "Turkish legal documents RAG system"}
            )
            print("❗ Yeni boş collection oluşturuldu.")

    def search_similar(self, query: str, n_results: int = 4, where: Optional[Dict] = None) -> Dict:
        """Benzer chunk'ları bul"""
        try:
            # Sorgu zenginleştirme
            enriched_query = self._enrich_query(query)
            
            # Daha fazla sonuç al, sonra filtrele
            results = self.collection.query(
                query_texts=[enriched_query],
                n_results=n_results * 4,
                where=where,
                include=["documents", "metadatas", "distances"]
            )
            
            print(f"\n🔍 Arama: '{query[:50]}...'")
            print(f"📊 Mesafe değerleri: {[f'{d:.3f}' for d in results['distances'][0][:4]]}")
            
            # Mesafe kontrolü
            is_large_distance = any(d > 10 for d in results['distances'][0][:3])
            
            if is_large_distance:
                print("⚠️ Büyük mesafe değerleri tespit edildi - yüzde bazlı filtreleme kullanılıyor")
                filtered_docs = results['documents'][0][:n_results]
                filtered_metas = results['metadatas'][0][:n_results]
                filtered_dists = results['distances'][0][:n_results]
            else:
                distance_threshold = 1.2
                filtered_docs = []
                filtered_metas = []
                filtered_dists = []
                
                for doc, meta, dist in zip(
                    results['documents'][0], 
                    results['metadatas'][0], 
                    results['distances'][0]
                ):
                    if dist <= distance_threshold and len(filtered_docs) < n_results:
                        filtered_docs.append(doc)
                        filtered_metas.append(meta)
                        filtered_dists.append(dist)
            
            print(f"✅ {len(filtered_docs)} sonuç bulundu")
            
            return {
                'query': query,
                'n_results': len(filtered_docs),
                'documents': filtered_docs,
                'metadatas': filtered_metas,
                'distances': filtered_dists,
            }
        except Exception as e:
            print(f"❌ Arama hatası: {e}")
            return {'query': query, 'n_results': 0, 'documents': [], 'metadatas': [], 'distances': []}
    
    def _enrich_query(self, query: str) -> str:
        """Sorguyu zenginleştir"""
        query_lower = query.lower()
        
        enrichments = {
            'cayma': 'cayma hakkı tüketici mesafeli sözleşme',
            'online alışveriş': 'mesafeli sözleşme internet alışveriş e-ticaret',
            'ayıplı': 'ayıplı mal kusurlu ürün garanti',
            'kredi': 'tüketici kredisi finansman borç',
            'iade': 'iade cayma geri verme',
            'garanti': 'garanti ayıp kusur',
        }
        
        for key, value in enrichments.items():
            if key in query_lower:
                return f"{query} {value}"
        
        return query

# ==============================================================================
# BÖLÜM 3: MERGED MODEL YÜKLEME (UNSLOTH YOK!)
# ==============================================================================

def load_merged_model(model_path: str):
    """
    Kaydedilmiş merged modeli direkt yükler.
    ❌ Unsloth kullanılmıyor
    ❌ LoRA adapter yüklenmiyor
    ✅ Sadece transformers kullanılıyor
    """
    print(f"\n{'='*70}")
    print(f"🔄 MERGED MODEL YÜKLENİYOR")
    print(f"📁 Path: {model_path}")
    print(f"{'='*70}\n")
    
    try:
        # 1. Tokenizer yükle
        print("📖 Tokenizer yükleniyor...")
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            print("⚙️  pad_token = eos_token olarak ayarlandı")
        
        print("✅ Tokenizer başarıyla yüklendi\n")
        
        # 2. Model yükle - 4-bit quantization
        print("🧠 Model yükleniyor (4-bit quantization)...")
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        
        model.eval()
        
        param_count = sum(p.numel() for p in model.parameters()) / 1e9
        print(f"✅ Model başarıyla yüklendi")
        print(f"📊 Parametre sayısı: ~{param_count:.2f}B")
        print(f"🎯 Inference modu: Aktif\n")
        
        return model, tokenizer
    
    except Exception as e:
        print(f"\n{'='*70}")
        print(f"❌ MODEL YÜKLEME HATASI")
        print(f"{'='*70}")
        print(f"Hata: {e}\n")
        print("Kontrol listesi:")
        print(f"  ✓ Model path doğru mu? → {model_path}")
        print(f"  ✓ Şu dosyalar var mı?")
        print(f"    - config.json")
        print(f"    - tokenizer_config.json")
        print(f"    - model*.safetensors")
        print(f"  ✓ GPU/CPU memory yeterli mi?")
        return None, None

# ==============================================================================
# BÖLÜM 4: YANIT TEMİZLEYİCİ
# ==============================================================================

def clean_llm_output(raw_output: str, query: str) -> str:
    """LLM çıktısını temizler"""
    if not raw_output or len(raw_output.strip()) < 10:
        return ""
    
    output = raw_output.strip()
    sentences = re.split(r'(?<=[.!?])\s+', output)
    
    clean_sentences = []
    found_start = False
    
    for sentence in sentences:
        if not found_start:
            if sentence and len(sentence) > 15 and sentence[0].isupper():
                valid_starts = [
                    'Evet', 'Hayır', 'Tüketici', 'Kredi', 'Online', 'Alışveriş', 
                    'Cayma', 'Ayıplı', 'Mal', 'Satıcı', 'Sözleşme', 'Kanun',
                    'İlgili', 'Bu', 'Söz', 'Mesafeli', 'Hak', 'Hakkınız',
                    '6502', 'Madde', 'Yönetmelik'
                ]
                
                if any(sentence.startswith(word) for word in valid_starts):
                    found_start = True
                    clean_sentences.append(sentence)
        else:
            clean_sentences.append(sentence)
    
    cleaned = ' '.join(clean_sentences).strip()
    
    if not cleaned or len(cleaned) < 30:
        if len(output) > 100:
            return output[100:].strip()
        return output
    
    return cleaned

# ==============================================================================
# BÖLÜM 5: RAG PIPELINE
# ==============================================================================

def generate_rag_response(query: str, llm_model, llm_tokenizer, vector_store, max_new_tokens: int = 512):
    """RAG ile yanıt üretir"""
    
    # 1. Retrieval
    search_results = vector_store.search_similar(query=query, n_results=4)
    
    if search_results['n_results'] == 0:
        yield "Üzgünüm, bu konuda elimde yeterli hukuki kaynak bulunamadı. Lütfen sorunuzu farklı şekilde ifade etmeyi deneyin.", []
        return

    # 2. Context Hazırlama
    contexts = []
    source_details = []
    
    for i, (doc, meta, dist) in enumerate(zip(
        search_results['documents'], 
        search_results['metadatas'],
        search_results['distances']
    )):
        source_info = f"{meta.get('file_name', 'Kaynak')}"
        if meta.get('article_number'):
            source_info += f" - Madde {meta['article_number']}"
        elif meta.get('section'):
            source_info += f" - {meta['section']}"
        
        doc_text = doc.strip()[:350] + "..." if len(doc.strip()) > 350 else doc.strip()
        
        contexts.append(f"Kaynak {i+1}: {doc_text}")
        source_details.append(f"[{i+1}] {source_info} (benzerlik: {dist:.2f})")

    context_block = "\n\n".join(contexts)

    # 3. Prompt
    system_prompt = """Sen Türk Tüketici Hukuku uzmanı bir asistansın. 

KURALLAR:
1. SADECE verilen kaynaklardaki bilgileri kullan
2. Kısa ve net yanıt ver (maksimum 4-5 cümle)
3. Önce ana cevabı ver, sonra detayları ekle
4. Gereksiz tekrar yapma
5. Kaynak numaralarını parantez içinde belirt

YANIT FORMATI:
[Evet/Hayır + Ana bilgi]. [Detay 1]. [Detay 2 (varsa)].

ÖRNEK:
"Evet, online alışverişte cayma hakkınız var (Kaynak 1). Ürünü teslim aldıktan sonra 14 gün içinde hiçbir neden göstermeden cayabilirsiniz (Kaynak 1). Bu hakkı kullanmak için satıcıya yazılı bildirimde bulunmanız yeterlidir (Kaynak 2)."
"""

    user_prompt = f"""KAYNAKLARDAN ALINAN BİLGİLER:

{context_block}

SORU: {query}

Yukarıdaki kaynaklara göre bu soruya kısa ve net yanıt ver. İlk cümlende "Evet" veya "Hayır" de."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    # 4. Tokenization
    input_ids = llm_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(DEVICE)

    # 5. Streamer
    streamer = TextIteratorStreamer(
        llm_tokenizer, 
        skip_prompt=True, 
        skip_special_tokens=True,
        timeout=30.0
    )

    # 6. Generation
    generation_kwargs = dict(
        input_ids=input_ids,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=0.3,
        top_p=0.85,
        top_k=40,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id,
        eos_token_id=llm_tokenizer.eos_token_id
    )

    thread = Thread(target=llm_model.generate, kwargs=generation_kwargs)
    thread.start()

    # 7. Stream
    raw_response = ""
    
    for new_text in streamer:
        raw_response += new_text
        
        if len(raw_response) < 80:
            continue
        
        cleaned_response = clean_llm_output(raw_response, query)
        
        if cleaned_response and len(cleaned_response) > 20:
            yield cleaned_response, source_details
    
    final_cleaned = clean_llm_output(raw_response, query)
    
    if final_cleaned and len(final_cleaned) > 20:
        yield final_cleaned, source_details
    else:
        yield raw_response.strip(), source_details
    
    thread.join()

# ==============================================================================
# BÖLÜM 6: GRADIO ARAYÜZÜ
# ==============================================================================

print("\n" + "="*70)
print("🚀 HUKUK PUSULASI SİSTEMİ BAŞLATILIYOR")
print("="*70 + "\n")

# Model ve VectorStore yükle
llm_model, llm_tokenizer = load_merged_model(MODEL_PATH)
vector_store = LegalVectorStore(CHROMADB_PERSIST_DIR, CHROMADB_COLLECTION_NAME, EMBEDDING_MODEL_NAME)


def gradio_chat_interface(message, history):
    """Gradio chat interface"""
    
    if llm_model is None or llm_tokenizer is None:
        yield "❌ Hata: Model yüklenemedi. Lütfen konsol loglarını kontrol edin."
        return
    
    response_generator = generate_rag_response(
        message, 
        llm_model, 
        llm_tokenizer, 
        vector_store,
        max_new_tokens=512
    )
    
    final_response = ""
    sources = []
    
    for partial_text, current_sources in response_generator:
        final_response = partial_text
        sources = current_sources
        yield partial_text
    
    # Kaynakları ekle
    if sources:
        source_block = "\n\n" + "─" * 50 + "\n**📚 Kullanılan Kaynaklar:**\n" + "\n".join([f"• {s}" for s in sources])
        yield final_response + source_block
    else:
        yield final_response


# Arayüz
if llm_model and vector_store:
    
    demo = gr.ChatInterface(
        fn=gradio_chat_interface,
        examples=[
            "Online alışverişte cayma hakkım var mı? Ne kadar sürem var?",
            "Ayıplı mal durumunda tüketici olarak ne talep edebilirim?",
            "Tüketici kredisi çekerken dikkat etmem gereken hukuki konular nelerdir?",
            "Kapıdan satışlarda tüketici hakları nelerdir?"
        ],
        title="⚖️ Hukuk Pusulası - RAG Destekli Hukuki Asistan",
        description=f"""
        **Mimari:** Fine-tuned Llama-3-8B (Merged Model) + Türkçe BERT + ChromaDB RAG
        
        ---
        ⚠️ **Önemli Uyarı:** Bu bir yapay zeka asistanıdır. Sunduğu bilgiler sadece bilgilendirme amaçlıdır 
        ve kesinlikle resmi hukuki tavsiye yerine geçmez. Hukuki sorunlarınız için mutlaka bir avukata danışın.
        
        **Cihaz:** {DEVICE} | **Model:** Llama-3-8B (Merged & Quantized)
        """,
        theme="soft",
        submit_btn="📤 Gönder"
    )

    print("\n" + "="*70)
    print("✅ SİSTEM HAZIR - GRADIO BAŞLATILIYOR")
    print("="*70 + "\n")
    
    demo.launch(share=True, debug=True)
    
else:
    print("\n" + "="*70)
    print("❌ HATA: Sistem başlatılamadı")
    print("="*70)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


🖥️  Kullanılan cihaz: mps

🚀 HUKUK PUSULASI SİSTEMİ BAŞLATILIYOR


🔄 MERGED MODEL YÜKLENİYOR
📁 Path: /Users/beyzaasan/Projects/HukukPusulasi/merged_model_hukuk

📖 Tokenizer yükleniyor...

❌ MODEL YÜKLEME HATASI
Hata: data did not match any variant of untagged enum ModelWrapper at line 1251003 column 3

Kontrol listesi:
  ✓ Model path doğru mu? → /Users/beyzaasan/Projects/HukukPusulasi/merged_model_hukuk
  ✓ Şu dosyalar var mı?
    - config.json
    - tokenizer_config.json
    - model*.safetensors
  ✓ GPU/CPU memory yeterli mi?


ValueError: The sentence_transformers python package is not installed. Please install it with `pip install sentence_transformers`

In [16]:
import os
print("HF Cache Directory:", os.environ.get("HF_HOME", os.path.join(os.path.expanduser("~"), ".cache/huggingface")))

HF Cache Directory: /Users/beyzaasan/.cache/huggingface


In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import chromadb
from chromadb.utils import embedding_functions
from typing import List, Dict, Optional
import json
from datetime import datetime

class LegalRAGSystem:
    """
    Türkçe Hukuki RAG Sistemi
    
    Bileşenler:
    1. Merged LLaMA Model (Fine-tuned)
    2. ChromaDB Vector Store (Turkish BERT embeddings)
    3. Retrieval + Generation Pipeline
    """
    
    def __init__(
        self,
        model_path: str = "/Users/beyzaasan/Projects/HukukPusulasi/merged_model_hukuk",
        chroma_db_path: str = "/Users/beyzaasan/Projects/HukukPusulasi/legal_chroma_db",
        collection_name: str = "legal_documents_v2",
        embedding_model: str = "emrecan/bert-base-turkish-cased-mean-nli-stsb-tr",
        load_in_4bit: bool = True,
        device_map: str = "auto"
    ):
        """
        Args:
            model_path: Merged LLaMA modelinin yolu
            chroma_db_path: ChromaDB persist directory
            collection_name: ChromaDB collection ismi
            embedding_model: Embedding modeli (retrieval için)
            load_in_4bit: 4-bit quantization kullan (memory tasarrufu)
            device_map: Device mapping stratejisi
        """
        self.model_path = model_path
        self.chroma_db_path = chroma_db_path
        self.collection_name = collection_name
        
        print("=" * 70)
        print("HUKUK RAG SİSTEMİ BAŞLATILIYOR")
        print("=" * 70)
        
        # 1. LLaMA Model ve Tokenizer yükle
        print("\n📚 1. LLaMA Model yükleniyor...")
        self._load_llama_model(load_in_4bit, device_map)
        
        # 2. ChromaDB Vector Store yükle
        print("\n🔍 2. ChromaDB Vector Store yükleniyor...")
        self._load_vector_store(embedding_model)
        
        print("\n✅ RAG Sistemi hazır!\n")
    
    def _load_llama_model(self, load_in_4bit: bool, device_map: str):
        """Merged LLaMA modelini yükle"""
        try:
            # Tokenizer
            print(f"   Tokenizer yükleniyor: {self.model_path}")
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_path,
                trust_remote_code=True
            )
            
            # Padding token ayarla (LLaMA için gerekli)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
                self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
            
            # Model config
            if load_in_4bit:
                print("   4-bit quantization ile model yükleniyor...")
                quantization_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4"
                )
                
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_path,
                    quantization_config=quantization_config,
                    device_map=device_map,
                    trust_remote_code=True,
                    torch_dtype=torch.float16
                )
            else:
                print("   Full precision model yükleniyor...")
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_path,
                    device_map=device_map,
                    trust_remote_code=True,
                    torch_dtype=torch.float16
                )
            
            self.model.eval()
            print(f"   ✅ Model yüklendi: {self.model.config.model_type}")
            print(f"   Device: {self.model.device}")
            
        except Exception as e:
            print(f"   ❌ Model yükleme hatası: {e}")
            raise
    
    def _load_vector_store(self, embedding_model: str):
        """ChromaDB vector store'u yükle"""
        try:
            # Device seçimi
            if torch.cuda.is_available():
                device = "cuda"
            elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
                device = "mps"
            else:
                device = "cpu"
            
            print(f"   Embedding device: {device}")
            
            # ChromaDB client
            self.chroma_client = chromadb.PersistentClient(
                path=self.chroma_db_path
            )
            
            # Embedding function
            self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name=embedding_model,
                device=device
            )
            
            # Collection yükle
            self.collection = self.chroma_client.get_collection(
                name=self.collection_name,
                embedding_function=self.embedding_function
            )
            
            count = self.collection.count()
            print(f"   ✅ Vector store yüklendi: {count} dokümanlı")
            
        except Exception as e:
            print(f"   ❌ Vector store yükleme hatası: {e}")
            raise
    
    def retrieve_relevant_chunks(
        self,
        query: str,
        n_results: int = 5,
        where: Optional[Dict] = None,
        min_similarity: float = 0.3
    ) -> List[Dict]:
        """
        Query'ye göre ilgili chunk'ları getir
        
        Args:
            query: Kullanıcı sorusu
            n_results: Maksimum sonuç sayısı
            where: Metadata filtreleri (örn: {"doc_type": "regulation"})
            min_similarity: Minimum similarity threshold
        
        Returns:
            İlgili chunk'ların listesi
        """
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where,
            include=["documents", "metadatas", "distances"]
        )
        
        # Sonuçları formatla ve filtrele
        relevant_chunks = []
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        ):
            similarity = 1 - dist  # Cosine distance -> similarity
            
            if similarity >= min_similarity:
                relevant_chunks.append({
                    'content': doc,
                    'metadata': meta,
                    'similarity': similarity
                })
        
        return relevant_chunks
    
    def create_rag_prompt(
        self,
        query: str,
        relevant_chunks: List[Dict],
        system_prompt: Optional[str] = None
    ) -> str:
        """
        RAG prompt oluştur (context + query)
        
        Args:
            query: Kullanıcı sorusu
            relevant_chunks: İlgili dokümanlara
            system_prompt: Sistem promptu (opsiyonel)
        
        Returns:
            Formatlanmış prompt
        """
        if system_prompt is None:
            system_prompt = """Sen Türk hukuku konusunda uzman bir hukuk asistanısın. 
Sana verilen hukuki dokümanlara dayanarak soruları yanıtlıyorsun.
Yanıtların:
- Sadece verilen kaynaklara dayalı olmalı
- Açık ve anlaşılır olmalı
- İlgili madde/karar numaralarını içermeli
- Eğer bilgi yoksa, bunu açıkça belirtmeli"""
        
        # Context oluştur
        context_parts = []
        for i, chunk in enumerate(relevant_chunks, 1):
            meta = chunk['metadata']
            
            # Kaynak bilgisi
            source_info = f"Kaynak {i}"
            if meta.get('doc_type') == 'regulation':
                if meta.get('article_number'):
                    source_info += f" (Yönetmelik - Madde {meta['article_number']})"
            elif meta.get('doc_type') == 'court_decision':
                if meta.get('section'):
                    source_info += f" (Mahkeme Kararı - {meta['section']})"
            
            context_parts.append(
                f"{source_info}:\n{chunk['content']}\n"
            )
        
        context = "\n---\n".join(context_parts)
        
        # Chat template kullan
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

İlgili Hukuki Belgeler:
{context}

Soru: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        
        return prompt
    
    def generate_answer(
        self,
        prompt: str,
        max_new_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 0.9,
        do_sample: bool = True
    ) -> str:
        """
        LLaMA modeli ile yanıt üret
        
        Args:
            prompt: Input prompt (RAG prompt)
            max_new_tokens: Maksimum yeni token sayısı
            temperature: Sampling sıcaklığı (düşük = deterministik)
            top_p: Nucleus sampling threshold
            do_sample: Sampling kullan (False = greedy)
        
        Returns:
            Üretilen yanıt
        """
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096  # LLaMA context window
        ).to(self.model.device)
        
        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=do_sample,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode (sadece yeni üretilen kısmı)
        generated_text = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        
        return generated_text.strip()
    
    def answer_question(
        self,
        query: str,
        n_results: int = 5,
        where: Optional[Dict] = None,
        min_similarity: float = 0.3,
        max_new_tokens: int = 512,
        temperature: float = 0.7,
        verbose: bool = True
    ) -> Dict:
        """
        RAG pipeline: Retrieve + Generate
        
        Args:
            query: Kullanıcı sorusu
            n_results: Kaç chunk retrieve edilecek
            where: Metadata filtreleri
            min_similarity: Minimum similarity threshold
            max_new_tokens: Max token for generation
            temperature: Generation temperature
            verbose: Detaylı log göster
        
        Returns:
            Yanıt dict'i
        """
        if verbose:
            print(f"\n{'='*70}")
            print(f"SORU: {query}")
            print(f"{'='*70}\n")
        
        # 1. RETRIEVAL
        if verbose:
            print("🔍 1. İlgili dokümanlara bakılıyor...")
        
        relevant_chunks = self.retrieve_relevant_chunks(
            query=query,
            n_results=n_results,
            where=where,
            min_similarity=min_similarity
        )
        
        if verbose:
            print(f"   ✅ {len(relevant_chunks)} ilgili chunk bulundu")
            for i, chunk in enumerate(relevant_chunks, 1):
                print(f"      {i}. Similarity: {chunk['similarity']:.3f} - "
                      f"{chunk['metadata'].get('doc_type', 'unknown')}")
        
        if not relevant_chunks:
            return {
                'query': query,
                'answer': "Üzgünüm, bu soruyla ilgili hukuki bir belge bulamadım.",
                'sources': [],
                'timestamp': datetime.now().isoformat()
            }
        
        # 2. PROMPT CREATION
        if verbose:
            print("\n📝 2. Prompt oluşturuluyor...")
        
        prompt = self.create_rag_prompt(query, relevant_chunks)
        
        if verbose:
            print(f"   Prompt uzunluğu: {len(prompt)} karakter")
        
        # 3. GENERATION
        if verbose:
            print("\n🤖 3. Yanıt üretiliyor...")
        
        answer = self.generate_answer(
            prompt=prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature
        )
        
        if verbose:
            print(f"   ✅ Yanıt üretildi ({len(answer)} karakter)\n")
            print(f"{'='*70}")
            print("YANIT:")
            print(f"{'='*70}")
            print(answer)
            print(f"{'='*70}\n")
        
        # 4. RESULT
        return {
            'query': query,
            'answer': answer,
            'sources': [
                {
                    'content': chunk['content'][:200] + "...",
                    'metadata': chunk['metadata'],
                    'similarity': chunk['similarity']
                }
                for chunk in relevant_chunks
            ],
            'n_sources': len(relevant_chunks),
            'timestamp': datetime.now().isoformat()
        }
    
    def interactive_session(self):
        """İnteraktif soru-cevap oturumu"""
        print("\n" + "="*70)
        print("HUKUK RAG SİSTEMİ - İNTERAKTİF OTURUM")
        print("="*70)
        print("Türk hukuku hakkında sorular sorabilirsiniz.")
        print("Çıkmak için 'quit' veya 'exit' yazın.\n")
        
        while True:
            try:
                query = input("❓ Sorunuz: ").strip()
                
                if query.lower() in ['quit', 'exit', 'q']:
                    print("\n👋 Görüşmek üzere!")
                    break
                
                if not query:
                    continue
                
                result = self.answer_question(
                    query=query,
                    verbose=True
                )
                
                print(f"\n📚 Kaynaklar: {result['n_sources']} belge kullanıldı\n")
                
            except KeyboardInterrupt:
                print("\n\n👋 Görüşmek üzere!")
                break
            except Exception as e:
                print(f"\n❌ Hata: {e}\n")
    
    def save_conversation(self, results: List[Dict], output_file: str):
        """Konuşma geçmişini kaydet"""
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✅ Konuşma kaydedildi: {output_file}")


# KULLANIM ÖRNEKLERİ
if __name__ == "__main__":
    # RAG sistemi oluştur
    rag = LegalRAGSystem(
        model_path="/Users/beyzaasan/Projects/HukukPusulasi/merged_model_hukuk",
        chroma_db_path="/Users/beyzaasan/Projects/HukukPusulasi/legal_chroma_db",
        collection_name="legal_documents_v2",
        load_in_4bit=True
    )
    
    # ÖRNEK 1: Tek soru
    print("\n" + "="*70)
    print("ÖRNEK 1: Tek Soru")
    print("="*70)
    
    result1 = rag.answer_question(
        query="İş kazasında işverenin tazminat sorumluluğu nedir?",
        n_results=3,
        temperature=0.7
    )
    
    # ÖRNEK 2: Sadece yönetmeliklerde ara
    print("\n" + "="*70)
    print("ÖRNEK 2: Sadece Yönetmeliklerde Ara")
    print("="*70)
    
    result2 = rag.answer_question(
        query="Kişisel verilerin korunması için hangi önlemler alınmalıdır?",
        n_results=3,
        where={"doc_type": "regulation"},
        temperature=0.7
    )
    
    # ÖRNEK 3: Sadece mahkeme kararlarında ara
    print("\n" + "="*70)
    print("ÖRNEK 3: Sadece Mahkeme Kararlarında Ara")
    print("="*70)
    
    result3 = rag.answer_question(
        query="Tazminat miktarı nasıl hesaplanır?",
        n_results=3,
        where={"doc_type": "court_decision"},
        temperature=0.7
    )
    
    # ÖRNEK 4: İnteraktif oturum
    print("\n" + "="*70)
    print("ÖRNEK 4: İnteraktif Oturum Başlatılıyor")
    print("="*70)
    
    # İnteraktif oturum için uncomment et:
    # rag.interactive_session()
    
    # Konuşmaları kaydet
    results = [result1, result2, result3]
    rag.save_conversation(
        results,
        "/Users/beyzaasan/Projects/HukukPusulasi/rag_conversation.json"
    )

HUKUK RAG SİSTEMİ BAŞLATILIYOR

📚 1. LLaMA Model yükleniyor...
   Tokenizer yükleniyor: /Users/beyzaasan/Projects/HukukPusulasi/merged_model_hukuk
   ❌ Model yükleme hatası: data did not match any variant of untagged enum ModelWrapper at line 1251003 column 3


Exception: data did not match any variant of untagged enum ModelWrapper at line 1251003 column 3